# Notebook 2: Exploratory Data Analysis (EDA)
**Project:** Customer Churn Intelligence System — Telecom  
**Author:** Prathmesh Joshi

---
### Goal
Discover WHY customers are churning. Visualize patterns across contract type, tenure, charges, and payment method. Every chart here tells part of the story we'll present in the executive dashboard.

## Step 1: Import Libraries & Load Clean Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visual style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

# Load clean data from notebook 1
df = pd.read_csv('../data/telco_churn_clean.csv')
print(f'Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Overall churn rate: {df["Churn"].mean():.1%}')

## Step 2: Overall Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
churn_counts = df['Churn'].value_counts()
bars = axes[0].bar(['Stayed', 'Churned'], churn_counts.values,
                   color=['#4C72B0', '#DD8452'], width=0.5, edgecolor='white')
axes[0].set_title('Customer Count: Stayed vs Churned', fontsize=13, pad=12)
axes[0].set_ylabel('Number of Customers')
for bar, val in zip(bars, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=['Stayed (73.5%)', 'Churned (26.5%)'],
            colors=['#4C72B0', '#DD8452'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Rate Distribution', fontsize=13, pad=12)

plt.suptitle('Overall Churn Overview — 7,043 Customers', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../images/01_churn_overview.png', bbox_inches='tight')
plt.show()
print('Chart saved: images/01_churn_overview.png')

## Step 3: Churn by Contract Type
**Business question:** Does the type of contract a customer is on affect how likely they are to leave?

In [ ]:
contract_churn = df.groupby('Contract')['Churn'].agg(['mean', 'count']).reset_index()
contract_churn.columns = ['Contract', 'Churn Rate', 'Customer Count']
contract_churn['Churn Rate %'] = (contract_churn['Churn Rate'] * 100).round(1)

print('Churn rate by contract type:')
print(contract_churn.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#DD8452' if r > 30 else '#4C72B0' for r in contract_churn['Churn Rate %']]
bars = ax.bar(contract_churn['Contract'], contract_churn['Churn Rate %'],
              color=colors, width=0.5, edgecolor='white')

for bar, val in zip(bars, contract_churn['Churn Rate %']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}%', ha='center', fontsize=12, fontweight='bold')

ax.set_title('Churn Rate by Contract Type', fontsize=14, pad=12)
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Contract Type')
ax.set_ylim(0, 60)
ax.axhline(y=df['Churn'].mean()*100, color='gray', linestyle='--', alpha=0.7, label='Overall avg')
ax.legend()

plt.tight_layout()
plt.savefig('../images/02_churn_by_contract.png', bbox_inches='tight')
plt.show()
print('Chart saved: images/02_churn_by_contract.png')

## Step 4: Cohort Analysis — Churn by Tenure Bucket
**Business question:** At what stage of the customer lifecycle is churn highest?

In [ ]:
tenure_order = ['0-12 months', '13-24 months', '25-48 months', '49+ months']

tenure_churn = df.groupby('tenure_bucket')['Churn'].agg(['mean', 'count']).reset_index()
tenure_churn.columns = ['Tenure Bucket', 'Churn Rate', 'Count']
tenure_churn['Churn Rate %'] = (tenure_churn['Churn Rate'] * 100).round(1)
tenure_churn['Tenure Bucket'] = pd.Categorical(tenure_churn['Tenure Bucket'],
                                                 categories=tenure_order, ordered=True)
tenure_churn = tenure_churn.sort_values('Tenure Bucket')

print('Cohort churn analysis:')
print(tenure_churn[['Tenure Bucket','Count','Churn Rate %']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tenure_churn['Tenure Bucket'], tenure_churn['Churn Rate %'],
        marker='o', linewidth=2.5, markersize=8, color='#DD8452')

for i, row in tenure_churn.iterrows():
    ax.annotate(f"{row['Churn Rate %']}%\n({row['Count']:,} customers)",
                xy=(row['Tenure Bucket'], row['Churn Rate %']),
                xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=10)

ax.fill_between(tenure_churn['Tenure Bucket'], tenure_churn['Churn Rate %'],
                alpha=0.1, color='#DD8452')
ax.set_title('Cohort Churn Rate by Customer Tenure', fontsize=14, pad=12)
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Customer Tenure')
ax.set_ylim(0, 60)

plt.tight_layout()
plt.savefig('../images/03_cohort_churn_tenure.png', bbox_inches='tight')
plt.show()
print('Chart saved: images/03_cohort_churn_tenure.png')

## Step 5: Churn by Payment Method

In [ ]:
payment_churn = df.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=False)
payment_churn_pct = (payment_churn * 100).round(1)

print('Churn rate by payment method:')
print(payment_churn_pct)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#DD8452' if v > 30 else '#4C72B0' for v in payment_churn_pct.values]
bars = ax.barh(payment_churn_pct.index, payment_churn_pct.values,
               color=colors, height=0.5, edgecolor='white')

for bar, val in zip(bars, payment_churn_pct.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=11, fontweight='bold')

ax.set_title('Churn Rate by Payment Method', fontsize=14, pad=12)
ax.set_xlabel('Churn Rate (%)')
ax.set_xlim(0, 55)

plt.tight_layout()
plt.savefig('../images/04_churn_by_payment.png', bbox_inches='tight')
plt.show()
print('Chart saved: images/04_churn_by_payment.png')

## Step 6: Monthly Charges Distribution — Churned vs Stayed

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

churned = df[df['Churn'] == 1]['MonthlyCharges']
stayed = df[df['Churn'] == 0]['MonthlyCharges']

ax.hist(stayed, bins=40, alpha=0.6, color='#4C72B0', label=f'Stayed (n={len(stayed):,})', edgecolor='white')
ax.hist(churned, bins=40, alpha=0.7, color='#DD8452', label=f'Churned (n={len(churned):,})', edgecolor='white')

ax.axvline(stayed.mean(), color='#4C72B0', linestyle='--', linewidth=2,
           label=f'Stayed avg: ${stayed.mean():.0f}')
ax.axvline(churned.mean(), color='#DD8452', linestyle='--', linewidth=2,
           label=f'Churned avg: ${churned.mean():.0f}')

ax.set_title('Monthly Charges Distribution — Churned vs Stayed', fontsize=14, pad=12)
ax.set_xlabel('Monthly Charges ($)')
ax.set_ylabel('Number of Customers')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../images/05_monthly_charges_dist.png', bbox_inches='tight')
plt.show()
print('Chart saved: images/05_monthly_charges_dist.png')

## Step 7: Correlation Heatmap — What Drives Churn?

In [ ]:
# Select numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Correlation with Churn column — sorted
churn_corr = df[numeric_cols].corr()['Churn'].drop('Churn').sort_values()

fig, ax = plt.subplots(figsize=(9, 8))
colors = ['#DD8452' if v > 0 else '#4C72B0' for v in churn_corr.values]
bars = ax.barh(churn_corr.index, churn_corr.values, color=colors, height=0.6, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Customer Churn', fontsize=14, pad=12)
ax.set_xlabel('Correlation Coefficient')

for bar, val in zip(bars, churn_corr.values):
    xpos = bar.get_width() + 0.005 if val >= 0 else bar.get_width() - 0.005
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9,
            ha='left' if val >= 0 else 'right')

plt.tight_layout()
plt.savefig('../images/06_churn_correlation.png', bbox_inches='tight')
plt.show()
print('Chart saved: images/06_churn_correlation.png')

print('\nTop 5 factors MOST correlated with churn:')
print(churn_corr.tail(5).to_string())

## Step 8: EDA Summary

In [ ]:
print('=' * 55)
print('       EDA SUMMARY — KEY FINDINGS')
print('=' * 55)

mtm_churn = df[df['Contract'] == 'Month-to-month']['Churn'].mean()
new_churn  = df[df['tenure_bucket'] == '0-12 months']['Churn'].mean()
ec_churn   = df[df['PaymentMethod'] == 'Electronic check']['Churn'].mean()
autopay_churn = df[df['PaymentMethod'] == 'Bank transfer (automatic)']['Churn'].mean()
high_charge_churn = df[df['MonthlyCharges'] > 65]['Churn'].mean()

print(f'\n Overall churn rate:          {df["Churn"].mean():.1%}')
print(f' Month-to-month churn:         {mtm_churn:.1%}')
print(f' New customer churn (0-12mo):  {new_churn:.1%}')
print(f' Electronic check churn:       {ec_churn:.1%}')
print(f' Auto-pay churn:               {autopay_churn:.1%}')
print(f' High-charge (>$65) churn:     {high_charge_churn:.1%}')
print(f' Electronic check vs auto-pay: {ec_churn/autopay_churn:.1f}x higher churn rate')
print('=' * 55)

---
## ✅ Notebook 2 Complete

**Key insights discovered:**
- Month-to-month contracts have dramatically higher churn than annual/2-year plans
- New customers (0–12 months) are the most at-risk cohort
- Electronic check users churn at 2.4× the rate of auto-pay customers
- High monthly charges (>$65) without add-on services = high churn signal

**All charts saved to `images/` folder.**

**Next:** Open `03_churn_prediction_model.ipynb`